# Benchmark A: 切り出しスループット実測（bench_cutout_scale.ipynb）
**目的**: SOC シミュの全カタログを対象に切り出しを量産し、
**「10⁶ 切り出しあたりの時間と費用」**を実測する。

実行環境: いつもの EC2(m7i-flex.large)で OK。GPU 不要。
所要: 30 分前後 / 費用: $0.1 未満の見込み。

In [1]:
MY_BUCKET   = "YOUR-BUCKET-roman-scratch"   # ベンチ産物は scratch 側へ
ROMAN_PREFIX = "stpubdata/roman/nexus/soc_simulations/tutorial_data/roman-2026.2"
CUTOUT_SIZE  = 64
MAX_PER_CAT  = None      # None = カタログ内全天体。まず 500 で小試行→ None で本番も可
INSTANCE_USD_PER_H = 0.0958   # m7i-flex.large オンデマンド

In [2]:
import time, io, numpy as np, pandas as pd, s3fs, pyarrow.parquet as pq, asdf
fs_pub  = s3fs.S3FileSystem(anon=True)
fs_mine = s3fs.S3FileSystem()

cat_paths = sorted(fs_pub.glob(f"{ROMAN_PREFIX}/**/*_cat.parquet"))
print(len(cat_paths), "catalogs")

73 catalogs


In [3]:
def image_for(cat_path):
    """カタログに対応する画像(ASDF)を探す: L2 の _cal.asdf か L3 の coadd"""
    stem = cat_path.rsplit("_cat.parquet", 1)[0]
    for suffix in ("_cal.asdf", "_coadd.asdf"):
        p = stem + suffix
        if fs_pub.exists(p):
            return p
    hits = fs_pub.glob(stem + "*.asdf")
    return hits[0] if hits else None

def cut_catalog(cat_path):
    """1 カタログ分の切り出し。戻り値: (切り出し数, 秒, 内訳)"""
    t0 = time.perf_counter()
    tbl = pq.read_table("s3://" + cat_path, filesystem=fs_pub, columns=["ra", "dec"])
    df = tbl.to_pandas()
    if MAX_PER_CAT:
        df = df.head(MAX_PER_CAT)
    t_cat = time.perf_counter() - t0

    img_path = image_for(cat_path)
    if img_path is None:
        return 0, 0.0, "no-image"

    t1 = time.perf_counter()
    with fs_pub.open(img_path, "rb") as f, asdf.open(f, lazy_load=True) as af:
        img = np.asarray(af["roman"]["data"])
        wcs = af["roman"]["meta"]["wcs"]
    t_read = time.perf_counter() - t1

    t2 = time.perf_counter()
    from astropy.coordinates import SkyCoord
    import astropy.units as u
    xs, ys = wcs.world_to_pixel(SkyCoord(df.ra.values*u.deg, df.dec.values*u.deg))
    h = CUTOUT_SIZE // 2
    H, W = img.shape
    cuts = []
    for x, y in zip(np.atleast_1d(xs), np.atleast_1d(ys)):
        xi, yi = int(round(float(x))), int(round(float(y)))
        if h <= xi < W - h and h <= yi < H - h:
            cuts.append(img[yi-h:yi+h, xi-h:xi+h])
    stack = np.stack(cuts).astype(np.float32) if cuts else np.zeros((0,))
    t_cut = time.perf_counter() - t2

    # scratch バケットへ保存(I/O 込みの実測にする)
    t3 = time.perf_counter()
    name = cat_path.split("/")[-1].replace("_cat.parquet", "")
    buf = io.BytesIO(); np.savez_compressed(buf, cutouts=stack)
    with fs_mine.open(f"{MY_BUCKET}/bench_cutouts/{name}.npz", "wb") as f:
        f.write(buf.getvalue())
    t_save = time.perf_counter() - t3

    total = time.perf_counter() - t0
    return len(cuts), total, f"cat {t_cat:.1f} read {t_read:.1f} cut {t_cut:.1f} save {t_save:.1f}" 

In [4]:
results = []
t_all = time.perf_counter()
for i, p in enumerate(cat_paths):
    n, sec, detail = cut_catalog(p)
    results.append((p.split("/")[-1], n, sec))
    if i % 10 == 0 or n == 0:
        print(f"[{i+1}/{len(cat_paths)}] {p.split('/')[-1]}: {n} cuts in {sec:.1f}s ({detail})")
wall = time.perf_counter() - t_all
print(f"\nTOTAL: {sum(r[1] for r in results):,} cutouts in {wall/60:.1f} min")

[1/73] my_roman_mosaic_cat.parquet: 4489 cuts in 12.8s (cat 0.3 read 8.4 cut 0.1 save 3.9)
[11/73] r0003201001001001004_0001_wfi10_f106_cat.parquet: 1626 cuts in 5.5s (cat 0.1 read 3.9 cut 0.0 save 1.5)
[21/73] r0003201001001001004_0002_wfi02_f106_cat.parquet: 1488 cuts in 5.7s (cat 0.2 read 4.2 cut 0.0 save 1.3)
[31/73] r0003201001001001004_0002_wfi12_f106_cat.parquet: 1492 cuts in 5.1s (cat 0.1 read 3.7 cut 0.0 save 1.3)
[41/73] r0003201001001001004_0003_wfi04_f106_cat.parquet: 1486 cuts in 5.2s (cat 0.1 read 3.8 cut 0.0 save 1.3)
[51/73] r0003201001001001004_0003_wfi14_f106_cat.parquet: 1553 cuts in 5.7s (cat 0.1 read 4.1 cut 0.0 save 1.4)
[61/73] r0003201001001001004_0004_wfi06_f106_cat.parquet: 1401 cuts in 5.4s (cat 0.1 read 4.0 cut 0.0 save 1.2)
[71/73] r0003201001001001004_0004_wfi16_f106_cat.parquet: 1638 cuts in 5.7s (cat 0.1 read 4.1 cut 0.0 save 1.4)

TOTAL: 116,312 cutouts in 6.8 min


In [5]:
# --- 集計: スループットと単位あたりの費用 ---
n_total = sum(r[1] for r in results)
per_1e6_h  = wall / 3600 / n_total * 1e6
per_1e6_usd = per_1e6_h * INSTANCE_USD_PER_H
gb_stored = fs_mine.du(f"{MY_BUCKET}/bench_cutouts") / 2**30   # du() は合計バイト数(int)を返す

print(f"実測スループット : {n_total/wall:,.0f} cutouts/s")
print(f"10⁶ 切り出しあたり: {per_1e6_h:.2f} instance-h = ${per_1e6_usd:.3f}")
print(f"10⁷ 天体×4バンド  : {per_1e6_h*40:.1f} h = ${per_1e6_usd*40:.2f} (オンデマンド)")
print(f"保存サイズ        : {gb_stored:.2f} GiB / {n_total:,} cutouts")
print(f"→ 10⁷×4バンドの保管: {gb_stored/n_total*4e7:.0f} GiB ≈ ${gb_stored/n_total*4e7*0.023:.0f}/月")

実測スループット : 284 cutouts/s
10⁶ 切り出しあたり: 0.98 instance-h = $0.094
10⁷ 天体×4バンド  : 39.1 h = $3.75 (オンデマンド)
保存サイズ        : 1.50 GiB / 116,312 cutouts
→ 10⁷×4バンドの保管: 515 GiB ≈ $12/月
